# Comparaison de modèles IA


COPIER SA POUR METRE VOS IMAGE 

<div style="text-align: center; margin: 20px 0;">
    <img src="imagesNotebook/cnn.jpg" 
         style="width: 600px; border: 1px solid #ddd; border-radius: 8px; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
    <div style="font-style: italic; font-size: 0.9em; color: #555; margin-top: 8px;">
        Figure n : LEGENDE
    </div>
</div>

Ce notebook est dédié à l'analyse des nos différents modèles. 
Pour rappel, nos base de données sont les suivantes :
 - "Municipal Waste Management Cost Prediction"
 - "Garbage Classification".

Et notre problématique est : _Comment l’intelligence artificielle peut-elle soutenir une gestion intelligente des déchets en combinant la prévision des flux de déchets communaux et l’assistance au tri pour les citoyens à partir d’images ?_

L'objectif est de comparer nos différents modèles afin de savoir lequel est le plus performant.

##  Table des Matières
1. [Librairies et Configuration](#Librairies)
2. [Analyse des modèles IA de la Base de Données : Municipal Waste Management Cost Prediction](#bd1)
3. [Analyse des modèles IA de la Base de Données : Garbage Classification](#bd2)
4. [Conclusion](#conclu)

---

<a id="Librairies"></a>
## 1. Librairies et Configuration

In [1]:
%%capture
import pandas as pd
import numpy as np
import time
import random

import seaborn as sns
import matplotlib.pyplot as plt 
import matplotlib.image as mpimg
import geopandas as gpd
from sklearn.metrics import confusion_matrix


import tensorflow as tf
from tensorflow.keras import layers, models

import os

!pip install rpy2
%load_ext rpy2.ipython


file_path = "../data/public_data_waste_fee.csv"
df = pd.read_csv(file_path)

<a id="bd1"></a>
## 2. Analyse des modèles IA de la Base de Données : Municipal Waste Management Cost Prediction

<div style="margin-left: 30px; font-size: 25px;">
  2.1. Introduction
</div>
Pour la base de données quantitatives, nous avons décidé d'entraîner des modèles comme la régression linéaire, la régression logistique multinomiale et le Random Forest.

REGRESSION LINEAIRE

Si la régression logistique est traditionnellement utilisée pour des choix binaires (modèle de Bernoulli), notre étude s'appuie sur la régression logistique multinomiale. C'est une extension qui permet de gérer plusieurs catégories. Au lieu de prédire une probabilité de "présence ou absence", le modèle calcule un vecteur de probabilités pour 9 types de déchets différents, dont la somme est égale à 100%.

RANDOM FOREST

<div style="margin-left: 30px; font-size: 25px;">
  2.2. Méthodologie de comparaison des modèles
</div>

&nbsp;&nbsp;&nbsp;&nbsp; **a) Régression Linéaire Simple**

&nbsp;&nbsp;&nbsp;&nbsp; **b) Régression Logistique Multinomiale**

In [ ]:
%%R
library(nnet)
library(caret)

# Changer de répertoire de travail ! : Session --> Set Working Directory --> To source file location

set.seed(123)

df <- read.csv("../../Data/public_data_waste_fee.csv")
dechets_colonnes <- c("organic", "paper", "glass", "wood", "metal", "plastic", "raee", "texile", "other")
df[dechets_colonnes][is.na(df[dechets_colonnes])] <- 0

# On enlève les régions qui n'apparaissent qu'une seule fois
counts <- table(df$region)
regions_valides <- names(counts[counts >= 5])
df <- df[df$region %in% regions_valides, ]
df$region <- factor(df$region)

# réduction organic
df$vrai_label <- dechets_colonnes[max.col(df[, dechets_colonnes])]
df_equilibre <- df[df$vrai_label != "organic", ] 
df_organic_reduit <- df[df$vrai_label == "organic", ][1:200, ] 
df_final <- rbind(df_equilibre, df_organic_reduit)
df_final$region <- factor(df_final$region)

# Validation Croisée
k <- 5
# createFolds --> répartition des régions équitablement
folds_indices <- createFolds(df_final$region, k = k, list = TRUE)

erreurs_totale <- data.frame()

for(i in 1:k){
  test_indices <- folds_indices[[i]]
  train_cv <- df_final[-test_indices, ]
  test_cv  <- df_final[test_indices, ]
  
  Y_train_cv <- as.matrix(train_cv[, dechets_colonnes])
  
  model_cv <- multinom(Y_train_cv ~ log(pop) + gdp + wage + alt + pden + d_fee + sea + urb + finance + region, 
                       data = train_cv, trace = FALSE, maxit = 200)
  
  preds_probs <- predict(model_cv, newdata = test_cv, type = "probs")
  
  reels_probs <- as.matrix(test_cv[, dechets_colonnes])
  reels_probs <- reels_probs / rowSums(reels_probs) # On normalise pour que la somme = 1
  
  erreur_pli <- abs(reels_probs - preds_probs)
  erreurs_totale <- rbind(erreurs_totale, as.data.frame(erreur_pli))
}

rapport_erreurs <- data.frame(
  Classe = dechets_colonnes,
  Erreur_Moyenne_Points_Pct = round(colMeans(erreurs_totale, na.rm = TRUE) * 100, 2)
)
rmse_par_classe <- sqrt(colMeans(erreurs_totale^2, na.rm = TRUE)) * 100

rapport_complet <- data.frame(
  Classe = dechets_colonnes,
  MAE = rapport_erreurs$Erreur_Moyenne_Points_Pct,
  RMSE = round(rmse_par_classe, 2)
)

print(rapport_complet)

rmse_global <- sqrt(mean(as.matrix(erreurs_totale)^2, na.rm = TRUE)) * 100
cat("RMSE Global du modèle :", round(rmse_global, 2), "%\n")

# Exemple de la première ligne
exemple_mix <- predict(model_cv, newdata = df_final[1,], type = "probs")
print(round(exemple_mix * 100, 2))


UsageError: Cell magic `%%R` not found.


Pour la régression logistique multinomiale, nous avons essayé de prédire le taux de déchets par catégorie, avec le niveau de revenus par habitants (gdp), la densité de la population (pden), le nombre d’habitant (pop), l'altitude (alt) et l’indice d’urbanisation (urb). 
Cependant, la matrice de confusion nous montre que le modèle prédit beaucoup la catégorie organic.
Malgré une modification de la répartition des données (60/40), la matrice de confusion révèle que le modèle multinomial reste fortement biaisé en faveur de la catégorie Organic. Ce phénomène d'écrasement s'explique par le déséquilibre natif du dataset (classe organique ultra-majoritaire)

&nbsp;&nbsp;&nbsp;&nbsp; **c) Random Forest**

&nbsp;&nbsp;&nbsp;&nbsp; **d) Comparaison Random Forest et Régression linéaire multiple**

&nbsp;&nbsp;&nbsp;&nbsp; **e) Comparaison Random Forest et Régression Logistique Multinomiale**

Variables qualitatives

<div style="margin-left: 30px; font-size: 25px;">
  2.3. Description des Architectures
</div>


&nbsp;&nbsp;&nbsp;&nbsp; **a) Régression Logistique Multinomiale**


<div style="margin-left: 30px; font-size: 25px;">
  2.4 Résultats et Analyse
</div>

TABLEAU DES COMPARAISON 

In [3]:
MATRICE DE CONFUSION

y_true = []
y_pred = []

for images, labels in val_data:
    preds = model2.predict(images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

cm_cr = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_cr, annot=True, fmt='d', cmap='Purples',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matrice de Confusion - Modèle 1 (Décroissant)')
plt.ylabel('Classe réel')
plt.xlabel('Prédictions du Modèle')
plt.show()

SyntaxError: invalid syntax (3856093827.py, line 1)


<div style="text-align: center; margin: 20px 0;">
    <img src="imagesNotebook/resultat_RLM.png" 
         style="width: 600px; border: 1px solid #ddd; border-radius: 8px; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
    <div style="font-style: italic; font-size: 0.9em; color: #555; margin-top: 8px;">
        Figure n : Comparaison MAE et RMSE de la régression logistique multionomiale
    </div>
</div>


<div style="text-align: center; margin: 20px 0;">
    <img src="imagesNotebook/exemple_RLM.png" 
         style="width: 600px; border: 1px solid #ddd; border-radius: 8px; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
    <div style="font-style: italic; font-size: 0.9em; color: #555; margin-top: 8px;">
        Figure n : exemple du modèle de régression logistique multionomiale
    </div>
</div>

Le MAE (Mean Absolute Error) traite toutes les erreurs de la même manière. 
Par exemple, le modèle se trompe en moyenne de 5.45 points de pourcentage pour le papier.
Le RMSE (Root Mean Square Error) est beaucoup plus sévère.
Ici, le RMSE global est de 6.69%. On observe que le RMSE est toujours légérement supérieur au MAE, ce qui signifie que le modèle est stable.

<a id="bd2"></a>
## 3. Analyse des modèles IA de la Base de Données : Garbage Classification

<div style="margin-left: 30px; font-size: 25px;">
  3.1. Introduction
</div>

Pour préduire le type de dechets a partir d'une image, nous avons décider de creer et d'entrainer un réseaux de neuronne convolutif a partir de notre base de données d'images. Les réseauxx de neuronnes convolutivif dont Yann LeCun deveuloppa le premier réseaux fonctionnels (1998), provient des travaux de D. H. Hubel et T. N. Wiesel (1968) sur le cortex visuel du chat et des primates. Révolutionner par le model Alexnet en 2012, les réseaux de neurone convolutif sont aujourd'hui devenue la methode principale en classifaction d'image.

Pour préduire le type de dechets a partir d'une image, nous avons décider de creer et d'entrainer un réseaux de neuronne
 convolutif (CNN) a partir de notre base de données d'images. Les réseauxx de neuronnes convolutivif dont Yann LeCun
  deveuloppa le premier réseaux fonctionnels (1998), provient des travaux de D. H. Hubel et T. N. Wiesel (1968) 
  sur le cortex visuel du chat et des primates. Révolutionner par le model Alexnet en 2012,
 les réseaux de neurone convolutif sont aujourd'hui devenue la methode principale en classifaction d'image.

 Leur plus gros avantage aprenne eux meme eux-mêmes à identifier les caractéristiques importantes des images via les coucher
 Les réseaux de neurones convolutif sont également invariant à la transalation et conserve les relations spatiales. 

 Pour résumer, les réseaux de neurones convolutifs se composent de 3 types de couches : 
Une couche de convolution qui agit comme un filtre sur l’image pour extraire des caractéristiques (ligne verticale, horizontale, couleurs) via des opérations matricielles.
Une couche de pooling qui réduit la taille des images tout en gardant les informations importantes pour accélérer le traitement. On réduit les zones en prenant généralement la moyenne des pixels ou le maximum d’une zone donnée.
Une couche totalement connectée qui prend les caractéristiques extraites par les couches précédentes. Et donne la décision finale de la prédiction.




<style>
    .img-container {
        text-align: center;
        margin: 20px 0;
    }
    .custom-img {
        width: 600px;
        border: 1px solid #ddd;
        border-radius: 8px;
        box-shadow: 2px 2px 10px rgba(0,0,0,0.1);
    }
    .legende {
        font-style: italic;
        font-size: 0.9em;
        color: #555;
        margin-top: 8px;
    }
</style>

<div style="text-align: center; margin: 20px 0;">
    <img src="imagesNotebook/cnn.jpg" 
         style="width: 600px; border: 1px solid #ddd; border-radius: 8px; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
    <div style="font-style: italic; font-size: 0.9em; color: #555; margin-top: 8px;">
        Figure 1 : HAQUE, Kh Nafizul. "What is Convolutional Neural Network — CNN (Deep Learning)", LinkedIn, 3 avril 2023
    </div>
</div>

Un CNN fonctionne en deux étapes. Premièrement il enchaîne des couches de convolution et de pooling pour réduire les données. Puis ces informations sont aplaties et transmises à la couche totalement connectée qui prend la décision finale en renvoyant une probabilité.

Pour notre projet nous possédons déjà une base de données avec des images à une taille identique (512 x 384). Nous avons également deux dossiers distincts : un avec des images pour entraîner le modèle et l’autre avec d’autres images pour tester le modèle.
Notre but est donc de preduir le type de dechet d'une image quelquon parmis les classe suivante : Carton, Verre, Métal, Papier, Plastique et Ordures ménagères.

<div style="margin-left: 30px; font-size: 25px;">
  3.2. Méthodologie de comparaison des modèles
</div>

Afin de comparer de manière rigoureuse et équitable les différents modèles de réseaux de neurones convolutifs, nous avons mis un protocole expérimental contrôlé.

### Protocole expérimental

Tous les modèles sont entraînés dans des conditions identiques afin d’assurer la comparabilité des résultats. Les éléments suivants sont fixés :

* même jeu de données et mêmes partitions entre les ensemble d'entraînement de validation et de test
* mêmes hyperparamètres principaux : le nombre d’époques est fixées à 20, le batch size (nombre d'image par paquets) est fixé a 32, l'optimiseur choisis est Adam , learning rate
* même environnement matériel , tout les test on été réaliser via Google colab avec le T4 GPU pour des raisons de facilitée qu seins du groupe

Afin de prendre en compte la variabilité due aux facteurs aléatoires (initialisation des poids, ordre des données), chaque modèle est entraîné sur une même seed aléatoire.

Les seeds utilisées sont identiques pour tous les modèles, ce qui permet une comparaison équitable.

Nous avons  donc retenu une moyenne et un écart-type comme outils d'analyse pour chaque Modele. Cette approche permet de valider la performance du modèle tout en s'assurant de sa stabilité.

In [9]:
%%capture
seed = 25
epoch = 20
tf.keras.utils.set_random_seed(seed)
num_classes = 6 

base_path = f'../data'
img_height= 224
img_width = 224
batch_size = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    base_path +'/train',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True,
    seed = seed,
)

val_data = tf.keras.utils.image_dataset_from_directory(
   base_path +'/val',
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

test_data = tf.keras.utils.image_dataset_from_directory(
    base_path +'/test',
    image_size=(img_height, img_width),
    batch_size=batch_size
)
class_names = val_data.class_names


---
### Métriques d’évaluation

Pour comparer les performances de ces modèles nous allons utiliser plusieur critére d'évaluation: 


* **Accuracy (%)** sur le jeu de test. Elle est la métrique principale de comparaison *
* **Temps d’entraînement**
* **Nombre de paramètres**
* **Utilisation mémoire**

---

AJOUTER

Pour chaque modèle, les courbes suivantes sont analysées :

 évolution de la **loss**
 évolution de l’**accuracy** 
overfitting
underfitting
convergence

---

<div style="margin-left: 30px; font-size: 25px;">
  3.3. Description des Architectures
</div>



Pour nos modèles, nous avons utilisé la fonction d'activation ReLU.
La fonction d’activation permet au réseau de neurones convolutif d’apprendre des relations complexes en introduisant de la non-linéarité après chaque couche. 


<div style="text-align: center; margin: 20px 0;">
    <img src="imagesNotebook/ReLu.png" 
         style="width: 200px; border: 1px solid #ddd; border-radius: 8px; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
    <div style="font-style: italic; font-size: 0.9em; color: #555; margin-top: 8px;">
        Fonction ReLu
    </div>
</div>


La fonction d’activation ReLU présente plusieurs avantages. Elle permet d’accélérer l’apprentissage grâce à sa simplicité de calcul et en évitant que les gradients deviennent trop faibles. Enfin, elle rend le réseau plus efficace en supprimant les valeurs négatives, ce qui crée des neurones inactifs et simplifie l’apprentissage.

Pour tous les modèles nous avons utilisé l'optimisateur Adam.
Un optimiseur est un algorithme mathématique qui ajuste les poids d’un réseau de neurones en utilisant le gradient de la fonction de perte, afin de minimiser l’erreur du modèle. La velur par default du taux d’apprentissage d'Adam est de $0.001$
Pour cela, on calcule le gradient, puis on applique une méthode de descente de gradient. Le Gradient indique la direction et l'intensité de la plus forte pente de l'erreur. Puis L'optimiseur met a jours les poids en soustrayant une fraction du gradient aux valeurs actuels.
Adam est une version améliorer de la descente de gradient. Il utilise une estimations des moments du gradient afin de converger plus efficacement vers un minimum de la fonction de perte.

&nbsp;&nbsp;&nbsp;&nbsp; **a) Modèle avec couches en décroissant**

Pour notre premier modele nous avons chosie une structure en antonoire avec un nombre de filtre décroissant ($128 \to 64 \to 32 \to 16$).
L'idée etant de capturer un grand nombre de détails au début, puis dissiper cette information pour ne garder que les caractéristiques les plus discriminantes avant la classification.


In [2]:
start_time_dcr = time.time()

model_dcr = tf.keras.Sequential([
    layers.Rescaling(1./255),
    layers.Conv2D(128,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(16,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64,activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model_dcr.compile(optimizer='adam',
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=False),
  metrics=['accuracy'],)

callback_modele1 = tf.keras.callbacks.TensorBoard(log_dir="logs/model_dcr")

model_dcr.fit(
    train_data,
    validation_data=val_data,
    epochs=epoch,
    callbacks=[callback_modele1],
    verbose=0
)

end_time_dcr = time.time()

NameError: name 'time' is not defined

Pour commencder nous avons appliquée une normalisation des pixels afin de ramener les valeurs entre 0 et 1, ce qui facilite la convergence du modèle.

Concernant les couches de convolution, nous avons utilisé des noyaux de taille 4×4.
Le noyau est une matrice qui se déplace sur l’image afin d’extraire des caractéristiques locales. Une taille de noyau de 4 permet de capter des motifs relativement larges.
Après les couches de convolution et de pooling, nous utilisons une couche d'applatissement.
Cette couche transforme les matrices de caractéristiques en un vecteur 1D. Cela permet de passer des features spatiales à un vecteur exploitable par des couches entièrement connectées.
Cette dernier est composer de 64 neuronnes et de la fonction d'activation ReLu. 
Son rôle est de combiner les caractéristiques extraites précédemment pour apprendre des relations plus globales entre elles.
Comme expliquer précédemment la fonction ReLu permet permet d'ajouter de la non-linéarité au modèle pour apprendre des combinaisons complexes.

Entre les couches de convolution, nous avons ajouté des couches de pooling. Celles-ci permettent de réduire la dimension spatiale des cartes de caractéristiques tout en conservant les informations les plus pertinentes.
Dans notre cas, nous utilisons le MaxPooling, qui consiste à conserver la valeur maximale de chaque noyau.

Pour la Couche de classification nous avons choisis la fonction softmax.
Celle-ci transforme les sorties du réseau en probabilités. L’ensemble des sorties forme une distribution dont la somme est égale à 1.
Cela permet d’interpréter la sortie du modèle comme une répartition de probabilités sur les différentes classes, et ainsi de sélectionner la classe la plus probable.

La fonction de perte utilisée pour ce modele est SparseCategoricalCrossentropy.
La cross-entropy mesure l’écart entre les probabilités prédites par le modèle et les vraies classes.
Le paramètre from_logits = False signifie que les sorties du modèle sont déjà transformées en probabilités grâce à la fonction softmax.


&nbsp;&nbsp;&nbsp;&nbsp; **b) Modèle avec couches en croissant**

Pour ce deuxieme modele la structure est exactemetn la meme expter le nombre de filtre qui est ici croissant. Ce modèle suit une approche plus classique de vision par ordinateur où la complexité augmente avec la profondeur.
Il utilise 4 blocs de Convolution suivi d'une couche de MaxPooling. Le nombre de filtres double à chaque étape ($16 \to 32 \to 64 \to 128$). Augmenter le nombre de filtre au fure et a mesure permet de capturer des motifs de plus en plus complexes.

In [ ]:
start_time_cr = time.time()

model_cr = tf.keras.Sequential([
    layers.Rescaling(1./255),
    layers.Conv2D(16,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64,activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model_cr.compile(optimizer='adam',
              loss=tf.losses.SparseCategoricalCrossentropy(from_logits=False),
                metrics=['accuracy'],)

tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="logs/model_cr")

model_cr.fit(
    train_data,
    validation_data=val_data,
    epochs=epoch,
    callbacks=[tensorboard_callback],
    verbose=0
)
end_time_cr = time.time()

&nbsp;&nbsp;&nbsp;&nbsp; **c) Modèle AlexNet**

In [ ]:
start_time_alex = time.time()

model_alex = models.Sequential([
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    layers.Conv2D(filters=96, kernel_size=(11, 11), strides=(4, 4), activation='relu', padding='valid'),
    layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),
    layers.BatchNormalization(),
    layers.Conv2D(filters=256, kernel_size=(5, 5), strides=(1, 1), activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),
    layers.BatchNormalization(),
    layers.Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), activation='relu', padding='same'),
    layers.Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), activation='relu', padding='same'),
    layers.Conv2D(filters=256, kernel_size=(3, 3), strides=(1, 1), activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(4096, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4096, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model_alex.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_alex.summary()

epochs = 20
history = model_alex.fit(
    train_data,
    validation_data=val_data,
    epochs=epoch
)
end_time_alex = time.time()


&nbsp;&nbsp;&nbsp;&nbsp; **d) Modèle vggNet**

In [ ]:
start_time_vgg = time.time()

model_vgg = models.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    layers.Flatten(),
    layers.Dense(4096, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4096, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model_vgg.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,          # arrête si pas d'amélioration pendant 5 epoch
    restore_best_weights=True
)

start_time = time.time()

history = model_vgg.fit(  
    train_data,
    validation_data=val_data,
    epochs=epochs,
    callbacks=[early_stop]
)
end_time_vgg = time.time()


&nbsp;&nbsp;&nbsp;&nbsp; **e) Modèle Optimisé**

In [ ]:
start_time_opti = time.time()

model_opti = tf.keras.Sequential([
    layers.Rescaling(1./255),
    layers.Conv2D(32,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(256,4, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64,activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model_opti.compile(optimizer='adam',
              loss=tf.losses.SparseCategoricalCrossentropy(from_logits=False),
  metrics=['accuracy'],)

callback_modele3 = tf.keras.callbacks.TensorBoard(log_dir="logs/model_opti")

model_opti.fit(
    train_data,
  validation_data=val_data,
  epochs=10,
    callbacks=[callback_modele3]
)

end_time_opti = time.time()


<div style="margin-left: 30px; font-size: 25px;">
  3.4. Résultats et Analyse
</div>

In [ ]:
%%capture
results_dcr = model_dcr.evaluate(test_data)
results_dcr.append(end_time_dcr - start_time_dcr)
results_cr = model_cr.evaluate(test_data)
results_dcr.append(end_time_cr - start_time_cr)
results_alex = model_alex.evaluate(test_data)
results_dcr.append(end_time_alex - start_time_alex)
results_vgg = model_vgg.evaluate(test_data)
results_dcr.append(end_time_vgg - start_time_vgg)
results_opti = model_opti.evaluate(test_data)
results_dcr.append(end_time_opti - start_time_opti)
resulats=[results_dcr,results_cr,results_alex,results_vgg,results_opti]


79/79 ━━━━━━━━━━━━━━━━━━━━ 29s 364ms/step - accuracy: 0.2351 - loss: 1.7198
79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 79ms/step - accuracy: 0.9569 - loss: 0.1403


NameError: name 'model_alex' is not defined

<div style="margin-left: 30px; font-size: 25px;">
  3.5. Discussion / Conclusion
</div>

<a id="conclu"></a>
## 4. Conclusion